In [6]:
import numpy as np
import pandas as pd
import pickle
import torch
import torch.nn as nn

# 1. THE OLD MAP (Because this is what the PyTorch model was trained on!)
REVERSE_CLASS_MAP = {
    0: 'Human', 
    1: 'OpenAI', 
    2: 'Google', 
    3: 'Meta', 
    4: 'Anthropic'
}

# 2. Recreate your PyTorch Model Architecture
# (Make sure this exactly matches the structure from your tarefa3_pytorch.py!)
class TextClassifierDNN(nn.Module):
    def __init__(self, input_size, num_classes):
        super(TextClassifierDNN, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )
        
    def forward(self, x):
        return self.net(x)

# 3. Load the test dataset
df_teste = pd.read_csv('subm1.csv', sep=';')

# CRITICAL: Do NOT lemmatize or remove punctuation!
textos_teste = df_teste['Text'].fillna('')

# 4. Load Vectorizer & Transform
with open('tfidf_vectorizer.pkl', 'rb') as f:
    vectorizer = pickle.load(f)

X_teste_tabular = vectorizer.transform(textos_teste).toarray().astype(np.float32)

# 5. Load the PyTorch Model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
input_dim = X_teste_tabular.shape[1] # Should be 5000 based on our TF-IDF

# Initialize model and load weights (Make sure the .pth file name matches yours!)
modelo_pytorch = TextClassifierDNN(input_size=input_dim, num_classes=5).to(device)
modelo_pytorch.load_state_dict(torch.load('melhor_modelo_pytorch.pth', map_location=device, weights_only=True))
modelo_pytorch.eval()

# 6. Predict using PyTorch
X_tensor = torch.tensor(X_teste_tabular).to(device)
with torch.no_grad():
    outputs = modelo_pytorch(X_tensor)
    _, previsoes_idx = torch.max(outputs, 1)

previsoes_idx = previsoes_idx.cpu().numpy()

# 7. Map back to strings and Save
df_teste['Label'] = [REVERSE_CLASS_MAP[idx] for idx in previsoes_idx]
nome_ficheiro_saida = 'subm1-g6-MIA-B.csv'
df_teste.to_csv(nome_ficheiro_saida, index=False, sep=';')

print(f"Sucesso! Ficheiro {nome_ficheiro_saida} gerado.")
print("\nNova Distribuição de Previsões (PyTorch):")
print(df_teste['Label'].value_counts())

C:\Users\bruno\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator TfidfTransformer from version 1.7.2 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
C:\Users\bruno\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator TfidfVectorizer from version 1.7.2 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


Sucesso! Ficheiro subm1-g6-MIA-B.csv gerado.

Nova Distribuição de Previsões (PyTorch):
Label
Human        39
Meta         25
Google       20
Anthropic    11
OpenAI        5
Name: count, dtype: int64


In [7]:
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report

# 1. Load the ground truth and your predictions
df_real = pd.read_csv('subm1_labels_revealed.csv', sep=';')
df_subA = pd.read_csv('subm1-g6-MIA-B.csv', sep=';')

# (If you want to test B, just change the file name above!)

# 2. Merge them together using the ID column to ensure they line up perfectly
merged = pd.merge(df_subA, df_real, on='ID', suffixes=('_pred', '_real'))

# 3. Calculate the final accuracy score
accuracy = accuracy_score(merged['Label_real'], merged['Label_pred'])
print(f"✨ Final Accuracy Score: {accuracy * 100:.2f}% ✨\n")

# 4. Print the detailed breakdown to see the "Google Bias" in action
print("Detailed Breakdown (Precision & Recall):")
print(classification_report(merged['Label_real'], merged['Label_pred']))

✨ Final Accuracy Score: 73.00% ✨

Detailed Breakdown (Precision & Recall):
              precision    recall  f1-score   support

   Anthropic       0.82      0.53      0.64        17
      Google       0.70      0.82      0.76        17
       Human       0.79      0.91      0.85        34
        Meta       0.68      0.94      0.79        18
      OpenAI       0.40      0.14      0.21        14

    accuracy                           0.73       100
   macro avg       0.68      0.67      0.65       100
weighted avg       0.71      0.73      0.70       100

